# 29 — B1 supplemental subtitles + BGE-M3 (research-only)

Notebook này bổ sung feature subtitle cho đúng 5.301 keyframe B1 đã freeze. Nó **không chạy lại Whisper** và không ghi vào bundle BTC/live store. Mỗi frame mới kế thừa transcript của cửa sổ BTC `[keyframe hiện tại, keyframe BTC kế tiếp)`, xác định hoàn toàn bằng `frame_idx` đọc từ mapping CSV. Trường `pts_time` synthetic của B1 không được dùng.

Đầu ra nằm riêng tại `gate1-b1-subtitles-v1`: `subtitles.json`, `subtitles_dense.npz`, audit lineage và COMPLETE marker.

Quy trình: `PREFLIGHT` một video, sau đó Quang/Trọng chạy `SHARD` riêng, cuối cùng một người chạy `AGGREGATE`. GPU chỉ cần cho `PREFLIGHT`/`SHARD`; `AGGREGATE` chạy CPU.


In [ ]:
from pathlib import Path

RUNNER = "trong"                 # "quang" hoặc "trong"
RUNNERS = ("quang", "trong")
RUN_SCOPE = "PREFLIGHT"          # PREFLIGHT | SHARD | AGGREGATE
CONFIRM_SHARD = ""               # SHARD: đặt đúng bằng RUNNER
CONFIRM_AGGREGATE = False         # AGGREGATE: đổi True sau khi cả hai shard COMPLETE
SESSION_MAX_MINUTES = 600
BGE_BATCH_SIZE = 64
CANARY_MAX_ABS_TOLERANCE = 5e-4
CANARY_MIN_COSINE = 0.99999

REPO_URL = "https://github.com/AIVIETNAM-AIO-Triet-Descartes/multimodal-fusion-engine.git"
REPO_REF = "feature/quang/storage"
REPO_DIR = Path("/kaggle/working/multimodal-fusion-engine")
WORK_DIR = Path("/kaggle/working/b1-subtitle-producer")

SOURCE_ROOT = "/research/keyframe-augmentation/transnetv2-btc-b1-v1/gate1-b1-feature-aggregate-v1"
OUTPUT_ROOT = "/research/keyframe-augmentation/transnetv2-btc-b1-v1/gate1-b1-subtitles-v1"
EXPECTED_SOURCE_MANIFEST_PAYLOAD_HASH = "eb050dcb310b6c9940805d151efcb569d82218e9821e0af38c586d80d004308d"
EXPECTED_WORKLIST_SHA256 = "10aef1d16fb53bf7c49795858fe3846c3ca3f1619a1b1874225372a5711c569d"
EXPECTED_FRAMES = 5301
CHECKPOINT = "BAAI/bge-m3"
CANARY_VIDEO_ID = "L21_V001"

if RUNNER not in RUNNERS:
    raise ValueError(f"RUNNER phải thuộc {RUNNERS}")
if RUN_SCOPE not in {"PREFLIGHT", "SHARD", "AGGREGATE"}:
    raise ValueError("RUN_SCOPE phải là PREFLIGHT, SHARD hoặc AGGREGATE")
if RUN_SCOPE == "SHARD" and CONFIRM_SHARD != RUNNER:
    raise RuntimeError("SHARD bị khóa; đặt CONFIRM_SHARD đúng bằng RUNNER")
if RUN_SCOPE == "AGGREGATE" and not CONFIRM_AGGREGATE:
    raise RuntimeError("AGGREGATE bị khóa; chỉ đổi CONFIRM_AGGREGATE=True sau khi hai shard COMPLETE")
WORK_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
import os, subprocess, sys

def run(command, *, cwd=None, env=None):
    result = subprocess.run(command, cwd=cwd, env=env, check=False, text=True, capture_output=True)
    if result.returncode:
        raise RuntimeError(f"Command failed: {' '.join(command[:3])}\n{result.stderr[-1500:]}")
    return result.stdout.strip()

def refresh_repo():
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    askpass = WORK_DIR / "git-askpass.sh"
    askpass.write_text('#!/bin/sh\ncase "$1" in *Username*) printf x-access-token ;; *) printf %s "$GITHUB_TOKEN" ;; esac\n', encoding="utf-8")
    askpass.chmod(0o700)
    env = os.environ.copy()
    env.update({"GITHUB_TOKEN": token, "GIT_ASKPASS": str(askpass), "GIT_TERMINAL_PROMPT": "0"})
    try:
        if (REPO_DIR / ".git").is_dir():
            if run(["git", "status", "--porcelain"], cwd=REPO_DIR):
                raise RuntimeError("Checkout Kaggle có local changes; Restart Session rồi Run All")
            run(["git", "fetch", "--depth", "1", "origin", REPO_REF], cwd=REPO_DIR, env=env)
            run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=REPO_DIR)
        else:
            run(["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(REPO_DIR)], env=env)
    finally:
        askpass.unlink(missing_ok=True)

refresh_repo()
run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "notebooks/requirements-kaggle-btc-bge.txt")])
sys.path.insert(0, str(REPO_DIR / "src"))
print({"repo_commit": run(["git", "rev-parse", "HEAD"], cwd=REPO_DIR), "scope": RUN_SCOPE, "runner": RUNNER})


In [ ]:
import csv, gc, json, shutil, time
from collections import Counter, defaultdict
from datetime import datetime, timezone

import numpy as np
from kaggle_secrets import UserSecretsClient
from mfe.offline.bundle import BundleLayout
from mfe.offline.keyframe_augmentation.common import atomic_write_json, atomic_write_jsonl, read_jsonl, sha256_file, sha256_json
from mfe.offline.keyframe_augmentation.subtitle_alignment import align_supplemental_subtitles, build_btc_subtitle_windows
from mfe.offline.transport import ModalVolumeTransport

secrets = UserSecretsClient()
os.environ["MODAL_TOKEN_ID"] = secrets.get_secret("MODAL_TOKEN_ID")
os.environ["MODAL_TOKEN_SECRET"] = secrets.get_secret("MODAL_TOKEN_SECRET")

BTC = ModalVolumeTransport("keyframes-btc", create_if_missing=False)
RESEARCH = ModalVolumeTransport("keyframe-augmentation-research", create_if_missing=False)
BTC_LAYOUT = BundleLayout("aic_btc_b1_v1")
INPUT = WORK_DIR / "input"; INPUT.mkdir(parents=True, exist_ok=True)

source_complete_path = INPUT / "source-COMPLETE.json"
RESEARCH.get(f"{SOURCE_ROOT}/COMPLETE.json", source_complete_path)
source_complete = json.loads(source_complete_path.read_text(encoding="utf-8"))
unsigned = dict(source_complete); claimed = unsigned.pop("manifest_payload_hash", None)
if claimed != EXPECTED_SOURCE_MANIFEST_PAYLOAD_HASH or sha256_json(unsigned) != claimed:
    raise RuntimeError("Feature aggregate COMPLETE lineage drift")
if source_complete.get("status") != "COMPLETE" or source_complete.get("feature_frames") != EXPECTED_FRAMES:
    raise RuntimeError("Feature aggregate is not COMPLETE/5301")

worklist_path = INPUT / "b1-worklist.jsonl"
RESEARCH.get(f"{SOURCE_ROOT}/b1-worklist.jsonl", worklist_path)
if sha256_file(worklist_path) != EXPECTED_WORKLIST_SHA256:
    raise RuntimeError("B1 worklist SHA drift")
WORKLIST = list(read_jsonl(worklist_path))
if len(WORKLIST) != EXPECTED_FRAMES or len({row["keyframe_id"] for row in WORKLIST}) != EXPECTED_FRAMES:
    raise RuntimeError("B1 worklist count/identity mismatch")
BY_RUNNER = {name: [] for name in RUNNERS}
for row in WORKLIST:
    if row.get("runner") not in BY_RUNNER:
        raise RuntimeError(f"Unknown runner in worklist: {row.get('runner')}")
    BY_RUNNER[row["runner"]].append(row)
BY_VIDEO = defaultdict(list)
for row in BY_RUNNER[RUNNER]: BY_VIDEO[row["video_id"]].append(row)
print({"source": source_complete["status"], "frames": len(WORKLIST), "runner_frames": {k: len(v) for k,v in BY_RUNNER.items()}, "runner_videos": len(BY_VIDEO), "btc_mutated": False})


In [ ]:
MODEL = None
MODEL_REVISION = None
MODEL_SNAPSHOT_HASH = None
CANARY_REPORT = None

if RUN_SCOPE != "AGGREGATE":
    import torch
    from huggingface_hub import model_info, snapshot_download
    from sentence_transformers import SentenceTransformer
    if not torch.cuda.is_available():
        raise RuntimeError("PREFLIGHT/SHARD cần GPU; chọn Accelerator rồi Restart Session")
    MODEL_REVISION = model_info(CHECKPOINT).sha
    snapshot = Path(snapshot_download(CHECKPOINT, revision=MODEL_REVISION))
    model_files = {
        str(p.relative_to(snapshot)): sha256_file(p)
        for p in sorted(snapshot.rglob("*"))
        if p.is_file()
    }
    MODEL_SNAPSHOT_HASH = sha256_json(model_files)
    MODEL = SentenceTransformer(str(snapshot), device="cuda")

    canary_dir = WORK_DIR / "canary"; canary_dir.mkdir(parents=True, exist_ok=True)
    subtitle_path = canary_dir / "subtitles.json"; vector_path = canary_dir / "subtitles_dense.npz"
    BTC.get(BTC_LAYOUT.part("subtitles", CANARY_VIDEO_ID), subtitle_path)
    BTC.get(BTC_LAYOUT.part("subtitles_dense", CANARY_VIDEO_ID), vector_path)
    base_text = json.loads(subtitle_path.read_text(encoding="utf-8"))
    stored = np.load(vector_path, allow_pickle=False)
    stored_ids = stored["ids"].astype(str).tolist(); stored_by_id = {value: i for i,value in enumerate(stored_ids)}
    canary_id = next(value for value in sorted(base_text) if value in stored_by_id and str(base_text[value]).strip())
    observed = MODEL.encode([base_text[canary_id]], normalize_embeddings=True, batch_size=1, convert_to_numpy=True)[0].astype(np.float32)
    expected = stored["vectors"][stored_by_id[canary_id]].astype(np.float32)
    max_abs = float(np.max(np.abs(observed - expected)))
    cosine = float(np.dot(observed, expected) / (np.linalg.norm(observed) * np.linalg.norm(expected)))
    CANARY_REPORT = {"status": "PASS" if max_abs <= CANARY_MAX_ABS_TOLERANCE and cosine >= CANARY_MIN_COSINE else "BLOCKED", "keyframe_id": canary_id, "max_abs_diff": max_abs, "cosine": cosine, "max_abs_tolerance": CANARY_MAX_ABS_TOLERANCE, "min_cosine": CANARY_MIN_COSINE, "checkpoint": CHECKPOINT, "revision": MODEL_REVISION, "model_snapshot_hash": MODEL_SNAPSHOT_HASH}
    CANARY_REPORT["report_payload_hash"] = sha256_json(CANARY_REPORT)
    print({"BGE_SUBTITLE_CANARY": CANARY_REPORT, "gpu": torch.cuda.get_device_name(0)})
    if CANARY_REPORT["status"] != "PASS":
        raise RuntimeError("BGE subtitle numeric canary BLOCKED; không sinh delta vector")


In [ ]:
def remote_paths(video_id):
    return {"subtitles": f"{OUTPUT_ROOT}/parts/subtitles/{video_id}.json", "vectors": f"{OUTPUT_ROOT}/parts/subtitles_dense/{video_id}.npz", "audit": f"{OUTPUT_ROOT}/audits/{video_id}.jsonl", "report": f"{OUTPUT_ROOT}/reports/{video_id}.json", "marker": f"{OUTPUT_ROOT}/markers/{RUNNER}/{video_id}.json"}

def valid_marker(video_id):
    paths = remote_paths(video_id)
    if not RESEARCH.exists(paths["marker"]): return None
    local = WORK_DIR / "marker-check" / f"{video_id}.json"; local.parent.mkdir(parents=True, exist_ok=True)
    try:
        RESEARCH.get(paths["marker"], local); marker = json.loads(local.read_text(encoding="utf-8"))
        unsigned = dict(marker); payload_hash = unsigned.pop("marker_payload_hash", None)
        if payload_hash != sha256_json(unsigned): return None
        if marker.get("status") != "COMPLETE" or marker.get("runner") != RUNNER or marker.get("video_id") != video_id or marker.get("input_worklist_sha256") != EXPECTED_WORKLIST_SHA256: return None
        for name in ("subtitles", "vectors", "audit", "report"):
            if RESEARCH.sha256(paths[name]) != marker.get(f"{name}_sha256"): return None
        return marker
    except Exception:
        return None
    finally:
        local.unlink(missing_ok=True)

def process_video(video_id, *, publish):
    rows = sorted(BY_VIDEO[video_id], key=lambda row: int(row["frame_idx"]))
    local = WORK_DIR / "videos" / video_id; local.mkdir(parents=True, exist_ok=True)
    try:
        map_path = local / "map.csv"; base_subtitle_path = local / "base-subtitles.json"
        BTC.get(BTC_LAYOUT.map_keyframes(video_id), map_path)
        BTC.get(BTC_LAYOUT.part("subtitles", video_id), base_subtitle_path)
        with map_path.open("r", encoding="utf-8-sig", newline="") as stream: mapping_rows = list(csv.DictReader(stream))
        base_subtitles = json.loads(base_subtitle_path.read_text(encoding="utf-8"))
        windows = build_btc_subtitle_windows(video_id, mapping_rows)
        aligned = align_supplemental_subtitles(video_id, rows, windows, base_subtitles)
        ids = sorted(aligned.subtitles); texts = [aligned.subtitles[value] for value in ids]
        vectors = MODEL.encode(texts, normalize_embeddings=True, batch_size=BGE_BATCH_SIZE, convert_to_numpy=True).astype(np.float32) if ids else np.zeros((0,1024), dtype=np.float32)
        if vectors.shape != (len(ids), 1024) or not np.isfinite(vectors).all(): raise RuntimeError(f"{video_id}: invalid subtitle vectors {vectors.shape}")
        if ids and not np.allclose(np.linalg.norm(vectors, axis=1), 1.0, atol=1e-4): raise RuntimeError(f"{video_id}: subtitle vectors are not normalized")
        subtitle_out = local / "subtitles.json"; subtitle_out.write_text(json.dumps(aligned.subtitles, ensure_ascii=False, sort_keys=True, indent=1), encoding="utf-8")
        vector_out = local / "subtitles_dense.npz"; np.savez_compressed(vector_out, ids=np.asarray(ids, dtype=np.str_), vectors=vectors)
        audit_out = local / "audit.jsonl"; atomic_write_jsonl(audit_out, aligned.audit_rows)
        counts = Counter(row["status"] for row in aligned.audit_rows)
        report = {"schema_version": 1, "status": "COMPLETE", "video_id": video_id, "runner": RUNNER, "method": "inherit_btc_next_keyframe_window_by_frame_idx", "coordinate_source": "decoder_counter_frame_idx", "candidate_pts_time_used": False, "input_candidates": len(rows), "subtitle_rows": len(ids), "alignment_status_counts": dict(sorted(counts.items())), "base_map_sha256": sha256_file(map_path), "base_subtitles_sha256": sha256_file(base_subtitle_path), "input_worklist_sha256": EXPECTED_WORKLIST_SHA256, "checkpoint": CHECKPOINT, "model_revision": MODEL_REVISION, "model_snapshot_hash": MODEL_SNAPSHOT_HASH, "bge_canary_report_hash": CANARY_REPORT["report_payload_hash"], "heldout_accessed": False, "btc_mutated": False}
        report["report_payload_hash"] = sha256_json(report); report_out = local / "report.json"; atomic_write_json(report_out, report)
        artifacts = {"subtitles": subtitle_out, "vectors": vector_out, "audit": audit_out, "report": report_out}; hashes = {name: sha256_file(path) for name,path in artifacts.items()}
        if publish:
            remote = remote_paths(video_id)
            for name,path in artifacts.items():
                RESEARCH.put(path, remote[name])
                if RESEARCH.sha256(remote[name]) != hashes[name]: raise RuntimeError(f"{video_id}/{name}: remote hash mismatch")
            marker = {"schema_version": 1, "status": "COMPLETE", "runner": RUNNER, "video_id": video_id, "input_frames": len(rows), "subtitle_rows": len(ids), "input_worklist_sha256": EXPECTED_WORKLIST_SHA256, **{f"{name}_sha256": digest for name,digest in hashes.items()}, "heldout_accessed": False, "btc_mutated": False, "completed_at_utc": datetime.now(timezone.utc).isoformat()}
            marker["marker_payload_hash"] = sha256_json(marker); marker_path = local / "marker.json"; atomic_write_json(marker_path, marker)
            RESEARCH.put(marker_path, remote["marker"])
            if RESEARCH.sha256(remote["marker"]) != sha256_file(marker_path): raise RuntimeError(f"{video_id}: marker hash mismatch")
        return report
    finally:
        shutil.rmtree(local, ignore_errors=True); gc.collect()

if RUN_SCOPE in {"PREFLIGHT", "SHARD"}:
    completed = [video_id for video_id in sorted(BY_VIDEO) if valid_marker(video_id)] if RUN_SCOPE == "SHARD" else []
    pending = [video_id for video_id in sorted(BY_VIDEO) if video_id not in set(completed)]
    todo = pending[:1] if RUN_SCOPE == "PREFLIGHT" else pending
    failures = []; reports = []; started = time.perf_counter()
    for index, video_id in enumerate(todo, 1):
        if index > 1 and time.perf_counter() - started >= SESSION_MAX_MINUTES * 60: break
        try:
            report = process_video(video_id, publish=RUN_SCOPE == "SHARD"); reports.append(report)
            print({"done": video_id, "frames": report["input_candidates"], "subtitle_rows": report["subtitle_rows"], "status_counts": report["alignment_status_counts"]})
        except Exception as error:
            failures.append({"video_id": video_id, "error_type": type(error).__name__, "error": str(error)})
            print({"failed": failures[-1]})
            if RUN_SCOPE == "PREFLIGHT": raise
    if RUN_SCOPE == "PREFLIGHT":
        print({"PREFLIGHT": "PASS", "published": False, "videos": len(reports), "btc_mutated": False})
    else:
        verified = [video_id for video_id in sorted(BY_VIDEO) if valid_marker(video_id)]
        manifest = {"schema_version": 1, "status": "COMPLETE" if len(verified) == len(BY_VIDEO) else "PARTIAL", "runner": RUNNER, "assigned_videos": len(BY_VIDEO), "assigned_frames": len(BY_RUNNER[RUNNER]), "verified_videos": len(verified), "verified_frames": sum(len(BY_VIDEO[v]) for v in verified), "verified": verified, "failures_this_session": failures, "input_worklist_sha256": EXPECTED_WORKLIST_SHA256, "heldout_accessed": False, "btc_mutated": False, "created_at_utc": datetime.now(timezone.utc).isoformat()}
        manifest["manifest_payload_hash"] = sha256_json(manifest); path = WORK_DIR / f"runner-manifest-{RUNNER}.json"; atomic_write_json(path, manifest)
        remote = f"{OUTPUT_ROOT}/runner-manifests/{RUNNER}.json"; RESEARCH.put(path, remote)
        if RESEARCH.sha256(remote) != sha256_file(path): raise RuntimeError("runner manifest upload mismatch")
        print({"SHARD_STATUS": manifest["status"], "runner": RUNNER, "verified_videos": len(verified), "assigned_videos": len(BY_VIDEO), "verified_frames": manifest["verified_frames"], "assigned_frames": manifest["assigned_frames"]})


In [ ]:
if RUN_SCOPE == "AGGREGATE":
    aggregate_dir = WORK_DIR / "aggregate"; shutil.rmtree(aggregate_dir, ignore_errors=True); aggregate_dir.mkdir(parents=True)
    manifests = {}
    for runner in RUNNERS:
        path = aggregate_dir / f"runner-{runner}.json"; RESEARCH.get(f"{OUTPUT_ROOT}/runner-manifests/{runner}.json", path)
        manifest = json.loads(path.read_text(encoding="utf-8")); unsigned = dict(manifest); claimed = unsigned.pop("manifest_payload_hash", None)
        if claimed != sha256_json(unsigned) or manifest.get("status") != "COMPLETE" or manifest.get("input_worklist_sha256") != EXPECTED_WORKLIST_SHA256: raise RuntimeError(f"runner {runner} is not COMPLETE/compatible")
        manifests[runner] = manifest
    if sum(m["verified_frames"] for m in manifests.values()) != EXPECTED_FRAMES: raise RuntimeError("runner frame total mismatch")

    merged_text = {}; merged_audit = []; vector_ids = []; vector_chunks = []
    for runner in RUNNERS:
        for video_id in manifests[runner]["verified"]:
            base = aggregate_dir / "parts" / video_id; base.mkdir(parents=True, exist_ok=True)
            paths = {"subtitles": f"{OUTPUT_ROOT}/parts/subtitles/{video_id}.json", "vectors": f"{OUTPUT_ROOT}/parts/subtitles_dense/{video_id}.npz", "audit": f"{OUTPUT_ROOT}/audits/{video_id}.jsonl"}
            local = {name: base / ("subtitles_dense.npz" if name == "vectors" else f"{name}.jsonl" if name == "audit" else "subtitles.json") for name in paths}
            for name in paths: RESEARCH.get(paths[name], local[name])
            text = json.loads(local["subtitles"].read_text(encoding="utf-8")); overlap = set(merged_text) & set(text)
            if overlap: raise RuntimeError(f"duplicate subtitle ids: {sorted(overlap)[:3]}")
            merged_text.update(text); merged_audit.extend(read_jsonl(local["audit"]))
            stored = np.load(local["vectors"], allow_pickle=False); ids = stored["ids"].astype(str).tolist(); matrix = stored["vectors"].astype(np.float32)
            if ids != sorted(text) or matrix.shape != (len(ids), 1024): raise RuntimeError(f"{video_id}: text/vector parity mismatch")
            vector_ids.extend(ids); vector_chunks.append(matrix)
            shutil.rmtree(base)
    if len(merged_audit) != EXPECTED_FRAMES or len({row["keyframe_id"] for row in merged_audit}) != EXPECTED_FRAMES: raise RuntimeError("aggregate audit does not cover all B1 frames")
    ordered = np.argsort(np.asarray(vector_ids, dtype=np.str_)) if vector_ids else np.asarray([], dtype=np.int64)
    matrix = np.concatenate(vector_chunks, axis=0) if vector_chunks else np.zeros((0,1024), dtype=np.float32)
    vector_ids = [vector_ids[index] for index in ordered]; matrix = matrix[ordered]
    if vector_ids != sorted(merged_text): raise RuntimeError("aggregate subtitle/vector ID mismatch")
    if not np.isfinite(matrix).all() or (len(matrix) and not np.allclose(np.linalg.norm(matrix, axis=1), 1.0, atol=1e-4)): raise RuntimeError("aggregate subtitle vectors invalid")

    flat = aggregate_dir / "delta-flat/data/vectors"; flat.mkdir(parents=True)
    subtitles_out = flat / "subtitles.json"; subtitles_out.write_text(json.dumps(merged_text, ensure_ascii=False, sort_keys=True, indent=1), encoding="utf-8")
    vectors_out = flat / "subtitles_dense.npz"; np.savez_compressed(vectors_out, ids=np.asarray(vector_ids, dtype=np.str_), vectors=matrix)
    audit_out = aggregate_dir / "subtitle-alignment-audit.jsonl"; atomic_write_jsonl(audit_out, sorted(merged_audit, key=lambda row: row["keyframe_id"]))
    status_counts = Counter(row["status"] for row in merged_audit)
    report = {"schema_version": 1, "status": "PASS", "feature_frames": EXPECTED_FRAMES, "subtitle_rows": len(merged_text), "text_coverage": len(merged_text)/EXPECTED_FRAMES, "alignment_status_counts": dict(sorted(status_counts.items())), "method": "inherit_btc_next_keyframe_window_by_frame_idx", "candidate_pts_time_used": False, "input_worklist_sha256": EXPECTED_WORKLIST_SHA256, "runner_manifest_payload_hashes": {name: value["manifest_payload_hash"] for name,value in manifests.items()}, "heldout_accessed": False, "btc_mutated": False, "created_at_utc": datetime.now(timezone.utc).isoformat()}
    report["report_payload_hash"] = sha256_json(report); report_out = aggregate_dir / "aggregate-report.json"; atomic_write_json(report_out, report)
    uploads = {"delta-flat/data/vectors/subtitles.json": subtitles_out, "delta-flat/data/vectors/subtitles_dense.npz": vectors_out, "subtitle-alignment-audit.jsonl": audit_out, "aggregate-report.json": report_out}
    ledger = {}
    for relative,path in uploads.items():
        remote = f"{OUTPUT_ROOT}/{relative}"; digest = sha256_file(path); RESEARCH.put(path, remote)
        if RESEARCH.sha256(remote) != digest: raise RuntimeError(f"aggregate upload mismatch: {relative}")
        ledger[relative] = {"sha256": digest, "bytes": path.stat().st_size}
    complete = {"schema_version": 1, "status": "COMPLETE", "feature_frames": EXPECTED_FRAMES, "subtitle_rows": len(merged_text), "input_worklist_sha256": EXPECTED_WORKLIST_SHA256, "artifact_ledger": ledger, "aggregate_report_sha256": sha256_file(report_out), "heldout_accessed": False, "btc_mutated": False, "completed_at_utc": datetime.now(timezone.utc).isoformat()}
    complete["manifest_payload_hash"] = sha256_json(complete); complete_path = aggregate_dir / "COMPLETE.json"; atomic_write_json(complete_path, complete)
    RESEARCH.put(complete_path, f"{OUTPUT_ROOT}/COMPLETE.json")
    if RESEARCH.sha256(f"{OUTPUT_ROOT}/COMPLETE.json") != sha256_file(complete_path): raise RuntimeError("COMPLETE upload mismatch")
    print({"B1_SUBTITLES": "COMPLETE", "frames": EXPECTED_FRAMES, "subtitle_rows": len(merged_text), "coverage": report["text_coverage"], "status_counts": report["alignment_status_counts"], "remote": OUTPUT_ROOT, "btc_mutated": False})


## Điều kiện hoàn thành

- `PREFLIGHT: PASS` nghĩa là một video align đúng, BGE canary khớp và **không upload**.
- Mỗi runner chỉ hoàn thành khi `SHARD_STATUS: COMPLETE`. Rerun sẽ bỏ qua marker đã hash-verify.
- Chỉ chạy `AGGREGATE` sau khi cả Quang và Trọng COMPLETE. Marker cuối được ghi sau cùng.
- `source_window_silent` là khoảng không có lời hợp lệ, không được bịa text. `before_first_btc` phải hiện trong audit nếu tồn tại.
- Output vẫn là research delta; chưa được trỏ browser/index production vào trước review.
